# Geospatial AI Pneumonia Detection — Reconstructed Colab Notebook

This notebook reconstructs the methodology described in the MIGARS 2026 poster:
- Binary chest X-ray classification: `NORMAL` vs `PNEUMONIA`
- 224×224 resizing, normalization, and training-time augmentation
- Custom CNN + MobileNetV2 with CBAM attention
- Validation-tuned weighted probability fusion
- Grad-CAM explanation
- Optional regional hotspot map from a real region-level CSV

**Important:** This is a new, reproducible implementation based on the poster, not the original lost Colab notebook. It does not guarantee the poster's reported 93% / 91% / 92.5% metrics. Run it on the same dataset and split to evaluate honestly. This is an academic research prototype, not a clinical diagnostic tool.


In [ ]:
# 1. Environment and imports
!pip -q install scikit-learn matplotlib seaborn pandas folium

import os, json, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)
from sklearn.model_selection import ParameterGrid

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Connect Google Drive and set dataset folders

Expected folder structure (class folder names are case-insensitive):

```text
chest_xray/
  train/
    NORMAL/
    PNEUMONIA/
  val/                 # optional if you have a separate validation split
    NORMAL/
    PNEUMONIA/
  test/
    NORMAL/
    PNEUMONIA/
```

If you only have `train` and `test`, this notebook creates a validation split from `train`. Keep the test set untouched until final evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS to the folder containing train/ and test/
DATA_DIR = "/content/drive/MyDrive/chest_xray"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")
test_dir = os.path.join(DATA_DIR, "test")

if not os.path.isdir(train_dir) or not os.path.isdir(test_dir):
    raise FileNotFoundError(
        f"Could not find train/ and test/ inside {DATA_DIR}. "
        "Update DATA_DIR to your dataset folder."
    )

has_val = os.path.isdir(val_dir)


In [ ]:
# 3. Load images; label_mode='binary' gives NORMAL=0, PNEUMONIA=1
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, labels="inferred", label_mode="binary",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)

if has_val:
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, labels="inferred", label_mode="binary",
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
    )
else:
    # 20% validation split from train directory
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, labels="inferred", label_mode="binary",
        validation_split=0.2, subset="training",
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, labels="inferred", label_mode="binary",
        validation_split=0.2, subset="validation",
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False, seed=SEED
    )

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, labels="inferred", label_mode="binary",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

class_names = train_ds.class_names
print("Class order:", class_names)
if len(class_names) != 2:
    raise ValueError("This notebook expects exactly two class folders.")
if class_names[0].upper() not in ("NORMAL", "NORMALS"):
    print("WARNING: Confirm class order. Positive label is class index 1:", class_names[1])

# Cache/prefetch; avoid cache() if dataset is too large for available RAM.
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Collect validation labels once for model selection
y_val = np.concatenate([y.numpy().reshape(-1) for _, y in val_ds]).astype(int)
y_test = np.concatenate([y.numpy().reshape(-1) for _, y in test_ds]).astype(int)


In [ ]:
# 4. Augmentation and preprocessing
# Input images are RGB float values in [0,255]. MobileNetV2 preprocessing
# maps them to [-1,1]. Augmentation is applied only during training.
augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomFlip("horizontal"),
], name="augmentation")

def make_train_pipeline(ds):
    return ds.map(
        lambda x, y: (tf.keras.applications.mobilenet_v2.preprocess_input(
            augmentation(x, training=True)), y),
        num_parallel_calls=AUTOTUNE
    )

def make_eval_pipeline(ds):
    return ds.map(
        lambda x, y: (tf.keras.applications.mobilenet_v2.preprocess_input(x), y),
        num_parallel_calls=AUTOTUNE
    )

train_pp = make_train_pipeline(train_ds)
val_pp = make_eval_pipeline(val_ds)
test_pp = make_eval_pipeline(test_ds)


## 5. Model A — Custom CNN

The CNN learns spatial features from the X-ray images. Dropout and L2 regularization help reduce overfitting.


In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=(224, 224, 3), name="xray")
    x = inputs  # already normalized to [-1, 1] by the shared input pipeline
    x = layers.Conv2D(32, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="custom_conv2")(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="custom_conv3")(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="custom_conv4")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="Custom_CNN")

cnn = build_custom_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-4),
            loss="binary_crossentropy", metrics=["accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")])
cnn.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                      patience=2, min_lr=1e-7)
]
EPOCHS = 20
history_cnn = cnn.fit(train_pp, validation_data=val_pp,
                      epochs=EPOCHS, callbacks=callbacks)
cnn.save("/content/custom_cnn.keras")


## 6. Model B — MobileNetV2 + CBAM attention

MobileNetV2 is initialized with ImageNet weights. CBAM applies channel attention followed by spatial attention to the final feature map. The backbone is initially frozen for transfer learning.


In [ ]:
def cbam_block(feature_map, ratio=8, name="cbam"):
    channels = int(feature_map.shape[-1])

    # Channel attention: shared MLP over global average and max pooling
    shared_dense_1 = layers.Dense(max(channels // ratio, 8), activation="relu",
                                  kernel_initializer="he_normal",
                                  use_bias=True, name=f"{name}_mlp1")
    shared_dense_2 = layers.Dense(channels, kernel_initializer="he_normal",
                                  use_bias=True, name=f"{name}_mlp2")
    avg_pool = layers.GlobalAveragePooling2D(name=f"{name}_avgpool")(feature_map)
    max_pool = layers.GlobalMaxPooling2D(name=f"{name}_maxpool")(feature_map)
    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))
    channel = layers.Activation("sigmoid", name=f"{name}_channel_sigmoid")(
        layers.Add()([avg_out, max_out]))
    channel = layers.Reshape((1, 1, channels))(channel)
    x = layers.Multiply(name=f"{name}_channel_scale")([feature_map, channel])

    # Spatial attention: average/max across channels, then 7x7 convolution
    avg_spatial = layers.Lambda(lambda z: tf.reduce_mean(z, axis=-1, keepdims=True))(x)
    max_spatial = layers.Lambda(lambda z: tf.reduce_max(z, axis=-1, keepdims=True))(x)
    spatial = layers.Concatenate(axis=-1)([avg_spatial, max_spatial])
    spatial = layers.Conv2D(1, 7, padding="same", activation="sigmoid",
                            kernel_initializer="he_normal",
                            name=f"{name}_spatial_conv")(spatial)
    return layers.Multiply(name=f"{name}_spatial_scale")([x, spatial])

def build_mobilenet_cbam():
    inputs = keras.Input(shape=(224, 224, 3), name="xray")
    base = keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False,
        weights="imagenet", name="mobilenetv2_backbone"
    )
    base.trainable = False
    x = base(inputs, training=False)
    x = cbam_block(x, name="cbam")
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="MobileNetV2_CBAM")
    return model, base

mobile, mobile_base = build_mobilenet_cbam()
mobile.compile(optimizer=keras.optimizers.Adam(1e-4),
               loss="binary_crossentropy", metrics=["accuracy",
               keras.metrics.Precision(name="precision"),
               keras.metrics.Recall(name="recall")])
mobile.summary()

history_mobile = mobile.fit(train_pp, validation_data=val_pp,
                            epochs=EPOCHS, callbacks=callbacks)


In [ ]:
# Optional fine-tuning: unfreeze only the last 30 backbone layers.
# Run this after the initial transfer-learning stage if validation is stable.
mobile_base.trainable = True
for layer in mobile_base.layers[:-30]:
    layer.trainable = False

mobile.compile(optimizer=keras.optimizers.Adam(1e-5),
               loss="binary_crossentropy", metrics=["accuracy"])
history_finetune = mobile.fit(train_pp, validation_data=val_pp,
                              epochs=5, callbacks=callbacks)
mobile.save("/content/mobilenetv2_cbam.keras")


## 7. Weighted probability fusion

The poster describes weighted probability fusion but does not state the exact weights. This cell selects the weight using the validation set only, then freezes it for test evaluation. Class 1 is treated as the positive class.


In [ ]:
def predict_probabilities(model, ds):
    return model.predict(ds, verbose=1).reshape(-1)

cnn_val_prob = predict_probabilities(cnn, val_pp)
mobile_val_prob = predict_probabilities(mobile, val_pp)

best = None
for w_cnn in np.linspace(0, 1, 21):
    fused = w_cnn * cnn_val_prob + (1 - w_cnn) * mobile_val_prob
    pred = (fused >= 0.5).astype(int)
    score = f1_score(y_val, pred, zero_division=0)
    if best is None or score > best["f1"]:
        best = {"cnn_weight": float(w_cnn),
                "mobile_weight": float(1-w_cnn), "f1": float(score)}

W_CNN = best["cnn_weight"]
W_MOBILE = best["mobile_weight"]
print("Validation-selected weights:", best)

cnn_test_prob = predict_probabilities(cnn, test_pp)
mobile_test_prob = predict_probabilities(mobile, test_pp)
ensemble_test_prob = W_CNN*cnn_test_prob + W_MOBILE*mobile_test_prob
ensemble_test_pred = (ensemble_test_prob >= 0.5).astype(int)


In [ ]:
# 8. Final test metrics — report these actual results, not expected values
def show_metrics(y_true, prob, threshold=0.5, title="Model"):
    pred = (prob >= threshold).astype(int)
    print("\\n", title)
    print("Accuracy :", round(accuracy_score(y_true, pred), 4))
    print("Precision:", round(precision_score(y_true, pred, zero_division=0), 4))
    print("Recall   :", round(recall_score(y_true, pred, zero_division=0), 4))
    print("F1-score :", round(f1_score(y_true, pred, zero_division=0), 4))
    try: print("ROC-AUC  :", round(roc_auc_score(y_true, prob), 4))
    except ValueError: print("ROC-AUC  : unavailable")
    print(classification_report(y_true, pred, target_names=class_names,
                                zero_division=0))
    cm = confusion_matrix(y_true, pred)
    plt.figure(figsize=(5,4))
    plt.imshow(cm); plt.title(f"{title} — Confusion Matrix")
    plt.xticks([0,1], class_names); plt.yticks([0,1], class_names)
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    for (i,j), v in np.ndenumerate(cm): plt.text(j, i, str(v), ha="center", va="center")
    plt.colorbar(); plt.show()

show_metrics(y_test, ensemble_test_prob, title="Weighted CNN + MobileNetV2-CBAM Ensemble")


## 9. Grad-CAM explanation

This Grad-CAM implementation targets the final convolutional layer of a selected model. For MobileNetV2, `out_relu` is used as the final convolutional feature map. The overlay is an explanation of model attention, not proof that the model is clinically correct.


In [ ]:
def make_gradcam_heatmap(img_batch, model, last_conv_layer_name):
    grad_model = keras.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        class_channel = predictions[:, 0]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Select one image from test set
for image_batch, label_batch in test_ds.take(1):
    sample = image_batch[0:1]
    true_label = int(label_batch[0].numpy()[0])
    break

# The MobileNet model receives preprocessed [-1,1] inputs.
sample_pp = tf.keras.applications.mobilenet_v2.preprocess_input(sample)
heatmap = make_gradcam_heatmap(sample_pp, mobile, "out_relu")

plt.figure(figsize=(5,5))
plt.imshow(sample[0].numpy().astype("uint8"))
plt.imshow(tf.image.resize(heatmap[...,None], IMG_SIZE).numpy().squeeze(),
           cmap="jet", alpha=0.35)
plt.title(f"Grad-CAM | True: {class_names[true_label]} | "
          f"MobileNet probability: {float(mobile(sample_pp)[0][0]):.3f}")
plt.axis("off"); plt.show()


## 10. Optional geospatial hotspot mapping

The poster describes fusing validated predictions with regional spatial data. To produce a scientifically meaningful map, provide a real CSV containing region-level information. This notebook does **not** fabricate locations or case counts.

CSV columns required:
- `region`: district/region name
- `latitude`: numeric latitude
- `longitude`: numeric longitude
- `cases`: observed or validated case count / prevalence measure

If you have individual prediction records, aggregate them by region first and document how each prediction is linked to a region. Do not map a patient's location or identifiable health information.


In [ ]:
# Upload or point to your actual regional CSV.
# Example: REGION_CSV = "/content/drive/MyDrive/pneumonia_region_data.csv"
REGION_CSV = ""

if REGION_CSV and os.path.isfile(REGION_CSV):
    import folium
    from folium.plugins import HeatMap

    geo = pd.read_csv(REGION_CSV)
    required = {"region", "latitude", "longitude", "cases"}
    missing = required - set(geo.columns)
    if missing:
        raise ValueError(f"CSV is missing columns: {missing}")

    geo = geo.dropna(subset=["latitude", "longitude", "cases"]).copy()
    geo["cases"] = pd.to_numeric(geo["cases"], errors="coerce")
    geo = geo.dropna(subset=["cases"])
    center = [geo["latitude"].mean(), geo["longitude"].mean()]
    m = folium.Map(location=center, zoom_start=5)
    HeatMap(geo[["latitude", "longitude", "cases"]].values.tolist(),
            radius=20, blur=15).add_to(m)

    for _, r in geo.iterrows():
        folium.CircleMarker(
            location=[r["latitude"], r["longitude"]],
            radius=5, popup=f'{r["region"]}: {r["cases"]}',
            tooltip=str(r["region"])
        ).add_to(m)

    display(m)
    m.save("/content/pneumonia_hotspot_map.html")
    print("Saved: /content/pneumonia_hotspot_map.html")
else:
    print("No regional CSV supplied. Classification and Grad-CAM are complete; "
          "add your verified regional CSV to generate the hotspot map.")


In [ ]:
# 11. Save reproducibility information and model files
results = {
    "seed": SEED,
    "image_size": IMG_SIZE,
    "class_names": class_names,
    "cnn_weight": W_CNN,
    "mobilenet_cbam_weight": W_MOBILE,
    "test_accuracy": float(accuracy_score(y_test, ensemble_test_pred)),
    "test_precision": float(precision_score(y_test, ensemble_test_pred, zero_division=0)),
    "test_recall": float(recall_score(y_test, ensemble_test_pred, zero_division=0)),
    "test_f1": float(f1_score(y_test, ensemble_test_pred, zero_division=0))
}
with open("/content/experiment_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print("Saved model files: /content/custom_cnn.keras and /content/mobilenetv2_cbam.keras")
